# שיעור 16 - פריסה של סוכנים מדרגים עם Microsoft Foundry

במחברת זו תבנו **סוכן תמיכת לקוחות מוכן לייצור** עבור החברה הבדיונית **Contoso**. בניגוד לשיעורים הקודמים, הנקודה כאן אינה בלולאת ההסקת מסקנות של הסוכן — אלא כל מה שמסביב לה שעושה את הסוכן בטוח להפעלה בקנה מידה:

1. **קריאת כלים** — שאילתות הזמנות ויצירת כרטיסי תמיכה.
2. **RAG** — תשובות מדיניות מבסיס ידע.
3. **זיכרון** — זכירת הלקוח לאורך הפניות.
4. **ניתוב מודל** — שליחת בקשות פשוטות למודל קטן, ודרישות מורכבות למודל גדול.
5. **מטמון תגובות** — מענה לשאלות חוזרות ללא קריאת מודל.
6. **אישור אדם** — החזרים מעל סף מסוים נעצרים לחתימה.
7. **שער הערכה** — ערכת מבחן במצב לא מקוון שחוסמת שחרור לקוי.
8. **תצפית** — מעקב OpenTelemetry סביב כל בקשה.

כל חלק עצמאי ורץ בפני עצמו. קראו כל שורה — הפרימיטיבים לייצור שמורים במכוון כקטנים.


## הגדרה

לפני הרצת פנקס זה, וודא שיש לך:

1. **פרויקט Microsoft Foundry** עם מודל צ'אט מיושם (למשל `gpt-4.1-mini`).
2. **התחברת עם Azure CLI** — הרץ `az login` במסוף שלך.
3. **הגדרת משתני סביבה דרושים:**
   - `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Microsoft Foundry שלך.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהושם.

מקטע RAG משתמש ב**Azure AI Search** כאשר `AZURE_SEARCH_SERVICE_ENDPOINT` ו-`AZURE_SEARCH_API_KEY` מוגדרים, ובמקרה שלא, משתמש בחיפוש בזיכרון כדי שהפנקס ירוץ בלי משאב חיפוש.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. כלים

כלים לייצור עושים עבודה אמיתית מול מערכות ממשיות. כאן אנו מדמים מסד נתוני הזמנות ומערכת כרטוס עם פונקציות פייתון פשוטות. הדקורטור `@tool` חושף אותן לסוכן.

שימו לב ש-`issue_refund` משתמש ב-`approval_mode="always_require"` להחזרים מעל סף מסוים — זהו הפרימיטיב של אדם במעגל שאנו מפעילים מאוחר יותר.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — בסיס ידע מדיניות

שאלות מדיניות ("מהי תקופת ההחזרה שלכם?") יש לענות מהמקור הסמכותי, לא מזיכרון המודל. אנו עוטפים בסיס ידע קטן ככלי חיפוש.

בפרודקשן זה **Azure AI Search**; כאן אנו מספקים חיפוש מילות מפתח בזיכרון כך שהמחברת פועלת בכל מקום, ומתעדכנת ל-Azure AI Search אוטומטית כאשר משתני הסביבה קיימים.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. זיכרון

סוכן תמיכה ששוכח עם מי הוא מדבר הוא סוכן תמיכה רע. אנחנו שומרים מאגר פרופילים קטן לכל לקוח ומשתילים סיכום קצר בהוראות הסוכן. בסביבה פרודקשן זו שירות זיכרון (ראו שיעור 13); כאן מילון עושה את התבנית נראית.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. ניתוב מודל וזיכרון מטמון של תגובות

שני מנופים של עלות המחוברים לטיפול בבקשה אחת:

- **ניתוב**: מסווג היוריסטי זול מחליט האם בקשה זקוקה למודל הקטן או למודל הגדול.
- **זיכרון מטמון**: שאלות חוזרות מנורמלות מוגשות ישירות מהמטמון ללא קריאה למודל.

המסווג כאן פשוט בכוונה. בפרודקשן היית מאמת אותו מול התנועה ויכול להחליפו בנתב מודל של Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 ו-8. הסוכן, אישור האדם, והתצפית

עכשיו אנחנו מרכיבים את הסוכן מהכלים שצוינו למעלה ועוטפים כל בקשה בתוך פס של OpenTelemetry. הפונקציה `handle_support_request` היא מטפל הבקשות בפרודקשן: מטמון → ניתוב → מעקב → הרצה → מטמון.

אישור האדם מטופל על ידי המסגרת: מכיוון ש-`issue_refund` הוא `approval_mode="always_require"`, ההרצה נעצרת ומציגה בקשת אישור שאנו פותרים במפורש.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. שער הערכה

זהו שער השחרור מהשיעור: סט בדיקות לא מקוון מעריך את הסוכן, והפריסה מתקיימת רק אם שיעור ההצלחה חוצה סף מסוים. המעריך כאן הוא בדיקת חפיפה פשוטה של מילות מפתח כדי לשמור על המחברת עצמאית; בייצור תשתמש במערכת שיפוט מבוססת LLM או במעריך מסגרת (ראה שיעור 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## לשלב את זה יחד: שחרור מדומה

התא למטה מציג את כל הלולאה שהשיעור מתאר: להריץ את שער ההערכה, ורק "לפרוס" אם הוא עובר. זהו התבנית שתריץ ב-CI לפני קידום גרסת סוכן לשירות סוכן Foundry.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## סיכום

הרכבתם סוכן תמיכה בלקוחות מוכן לייצור עם כל דאגה תפעולית משולבת:

- **כלים, RAG וזיכרון** מעניקים לסוכן יכולת והקשר.
- **ניתוב מודלים ומטמון** שומרים על זמני תגובה ועלויות תחת שליטה.
- **אישור אנושי** מגן על פעולות בסיכון גבוה כמו החזרים כספיים גדולים.
- **שער ההערכה** חוסם שחרורים רעים לפני שהם נשלחים.
- **מעקב** מאפשר שכל בקשה תהיה ניתנת לצפייה.

### אתגר

הרחיבו סוכן זה ל:

1. **תמיכה במודלים מרובים** — הוסיפו שכבה שלישית של "הסקה" ונתבו אליה הסלמות/תלונות.
2. **הוספת שערי הערכה** — הרחיבו את `TEST_CASES` לכלול תרחישי אישור החזר כספי ואשרו שהשער תופס נסיגות.
3. **הוספת ניתוב שמבוסס על עלות** — עקבו אחרי עלות משוערת לכל בקשה (קטנה מול גדולה מול מזכרון מטמון) והדפיסו דוח עלות אחרי אצווה של שאילתות מעורבות.

בשיעור הבא תעשו את המסלול ההפוך ותפעילו סוכן כולו על המחשב שלכם עם Microsoft Foundry Local ו-Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
